In [49]:
# Session Setup
# -------------

# Boilerplate Modules
import os
import sys
import json

from dotenv import load_dotenv

import time

from tqdm import tqdm

# Modelus to Build Corpus from Corpora Scrapped from the Web
import wikipediaapi
import arxiv
from sec_edgar_downloader import Downloader
from firecrawl import FirecrawlApp


In [37]:
# Load variables from .env into the environment
load_dotenv()

True

In [39]:
firecrawl_api_key = os.getenv("FIRECRAWL_KEY")

Defining topics

In [25]:
pages = [
    "Attention_Is_All_You_Need", 
    "BERT_(language_model)", 
    "GPT-4", 
    "Retrieval-augmented_generation", 
    "LoRA_(machine learning)",
    "Snowflake_Inc.",
    "Apache_Airflow",
    "Apache_Spark",
    "Apache_Beam",
    "Apache_Kafka",
    "ChromaDB",
    "Vector_database",
    "Dask_(software)",
    "Embedding_(machine_learning)",
    "Money_laundering",
    "Data_analysis_for_fraud_detection",
    "Anti-Money_Laundering_Improvement_Act",
    "JPMorgan_Chase",
    "Capital_One",
    ]

In [42]:
# List of target documentation roots
doc_urls = [
    "https://airflow.apache.org/docs/",
    # "https://spark.apache.org/docs/latest/",
    # "https://docs.snowflake.com/en/",
    # "https://beam.apache.org/documentation/",
    # "https://kafka.apache.org/documentation/"
]

Wikipedia

In [7]:
# Setting up an user-agent
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rics-rag-project/1.0 (rzambrano@gmail.com)'
)

In [26]:

wikipedia_docs = []

for title in tqdm(pages, desc="Pulling from Wikipedia...", colour='green'):
    page = wiki.page(title)
    if page.exists():
        wikipedia_docs.append({"title": title, "text": page.text})
        print(f"Ok... {title} ({len(page.text)} chars)")
    else:
        print(f"Oh No!... {title} — not found")
    print("-" * 50)

Pulling from Wikipedia...:  11%|█         | 2/19 [00:00<00:03,  5.65it/s]

Ok... Attention_Is_All_You_Need (19247 chars)
--------------------------------------------------
Ok... BERT_(language_model) (17402 chars)
--------------------------------------------------


Pulling from Wikipedia...:  21%|██        | 4/19 [00:00<00:02,  5.63it/s]

Ok... GPT-4 (14513 chars)
--------------------------------------------------
Ok... Retrieval-augmented_generation (11462 chars)
--------------------------------------------------


Pulling from Wikipedia...:  32%|███▏      | 6/19 [00:00<00:02,  6.35it/s]

...

Ok... Money_laundering (33331 chars)
--------------------------------------------------
Ok... Data_analysis_for_fraud_detection (10261 chars)
--------------------------------------------------


Pulling from Wikipedia...:  95%|█████████▍| 18/19 [00:03<00:00,  6.16it/s]

Ok... Anti-Money_Laundering_Improvement_Act (4989 chars)
--------------------------------------------------
Ok... JPMorgan_Chase (67477 chars)
--------------------------------------------------


Pulling from Wikipedia...: 100%|██████████| 19/19 [00:03<00:00,  5.86it/s]

Ok... Capital_One (19368 chars)
--------------------------------------------------


In [28]:
wikipedia_docs[-1]

{'title': 'Capital_One',
 'text': 'Capital One Financial Corporation is an American bank holding company headquartered in Tysons, Virginia, with operations in the United States, Canada, and the United Kingdom. One of the largest banks in the United States, it is the largest issuer of credit cards in the United States, and is one of the largest car finance companies in the U.S. It owns the Discover Card, Diners Club, and Pulse payment networks. The company\'s three business segments are credit cards, consumer banking, and commercial banking. It has approximately 750 bank branches, including over 60 café style locations, and 7,000 ATMs in the United States. The company\'s corporate headquarters are in the Capital One Tower and its European headquarters are in Trent House, Nottingham.\nThe company is ranked 82nd on the Fortune 500.\nThe company helped pioneer the mass marketing of credit cards in the 1990s.\n\nHistory\n1994–2004\nRichard Fairbank and Nigel Morris developed the idea of usi

Arxiv

In [29]:
# Setting up an user-agent
arxiv_client = arxiv.Client()

In [30]:
max_arxiv_results = 2
arxiv_docs = []

for title in tqdm(pages, desc="Pulling from Arxiv...", colour='green'):
    query = title.replace("_", " ").lower()
    arxiv_result = arxiv.Search(query=query, max_results=max_arxiv_results)
    for result in arxiv_client.results(arxiv_result):
        if result:
            arxiv_docs.append({"title": result.title, "text": result.summary}) # Using only the abstract
            print(f"Ok... {result.title} ({len(result.summary)} chars)")
        else:
            print(f"Oh No!... something went wrong :'(")
        print("-" * 50)


Pulling from Arxiv...:   5%|▌         | 1/19 [00:00<00:12,  1.47it/s]

Ok... Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet (1121 chars)
--------------------------------------------------
Ok... "All You Need" is Not All You Need for a Paper Title: On the Origins of a Scientific Meme (848 chars)
--------------------------------------------------


Pulling from Arxiv...:  11%|█         | 2/19 [00:03<00:35,  2.07s/it]

Ok... The Road to Know-Where: An Object-and-Room Informed Sequential BERT for Indoor Vision-Language Navigation (1379 chars)
--------------------------------------------------
Ok... TinyBERT: Distilling BERT for Natural Language Understanding (1392 chars)
--------------------------------------------------


Pulling from Arxiv...:  16%|█▌        | 3/19 [00:07<00:44,  2.79s/it]

...

Ok... Anti-Money Laundering Systems Using Deep Learning (1786 chars)
--------------------------------------------------
Ok... Intelligent Anti-Money Laundering Solution Based upon Novel Community Detection in Massive Transaction Networks on Spark (972 chars)
--------------------------------------------------


Pulling from Arxiv...:  95%|█████████▍| 18/19 [01:05<00:03,  3.91s/it]

Ok... Cross comparison and modelling of Goldman Sachs, Morgan Stanley, JPMorgan Chase, Bank of America, and Franklin Resources (565 chars)
--------------------------------------------------
Ok... JEL: Applying End-to-End Neural Entity Linking in JPMorgan Chase (1450 chars)
--------------------------------------------------


Pulling from Arxiv...: 100%|██████████| 19/19 [01:09<00:00,  3.66s/it]

Ok... The Theory of Crowd Capital (795 chars)
--------------------------------------------------
Ok... A Micro-foundation of Social Capital in Evolving Social Networks (1285 chars)
--------------------------------------------------


In [31]:
arxiv_docs[-1]

{'title': 'A Micro-foundation of Social Capital in Evolving Social Networks',
 'text': 'A social network confers benefits and advantages on individuals (and on groups), the literature refers to these advantages as social capital. This paper presents a micro-founded mathematical model of the evolution of a social network and of the social capital of individuals within the network. The evolution of the network is influenced by the extent to which individuals are homophilic, structurally opportunistic, socially gregarious and by the distribution of types in the society. In the analysis, we identify different kinds of social capital: bonding capital, popularity capital, and bridging capital. Bonding capital is created by forming a circle of connections, homophily increases bonding capital because it makes this circle of connections more homogeneous. Popularity capital leads to preferential attachment: individuals who become popular tend to become more popular because others are more likely

SEC

In [20]:
# Setting up an user-agent
dl = Downloader("MyDataScienceProject", "rzambrano@gmail.com")

In [21]:
dl.get("10-K", "JPM", limit=2)

2

In [ ]:
dl.get("10-K", "COF", limit=2)

2

Firecrawl

In [40]:
# Initialize the app
app = FirecrawlApp(api_key=firecrawl_api_key)

In [ ]:
for url in doc_urls:
        print(f"Starting crawl for: {url}")

        response  = app.crawl(
            url, 
            "limit": 100,
            scrape_options={
                'formats': [
                'markdown',
                { 'type': 'json', 'schema': { 'type': 'object', 'properties': { 'title': { 'type': 'string' } } } }
                ],
                'proxy': 'auto',
                'max_age': 600000,
                'only_main_content': True
                }
        )

Testing embedding model

In [1]:
from sentence_transformers import SentenceTransformer

c:\Users\rzamb\Documents\rag-eval-stack\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

c:\Users\rzamb\Documents\rag-eval-stack\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rzamb\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5598.71it/s]


In [3]:
# Embed a list of text chunks
embeddings = model.encode(
    ["Silverspoon is an awesome manga", "I would like to have a warm beach near me"],
    normalize_embeddings=True,  # important for cosine similarity
    show_progress_bar=True
)

Batches: 100%|██████████| 1/1 [00:00<00:00, 28.40it/s]


In [4]:
embeddings.shape

(2, 384)

In [11]:
type(embeddings)

numpy.ndarray

In [6]:
# embeddings[0]

Testing langchain

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """They were the bestselling singles band in the world. They had awards, credibility, commercial success and creative freedom. Then they deleted their records, 
        erased themselves from musical history and burnt their last million pounds in a boathouse on the Isle of Jura. And they couldn't say why. \n\n 
        This is not just the story of The KLF. It is a book about Carl Jung, Alan Moore, Robert Anton Wilson, Ken Campbell, Dada, Situationism, Discordianism, magic, chaos, punk, rave, the 
        alchemical symbolism of Doctor Who and the special power of the number 23. \n\n
        Wildly unauthorised and unlike any other music biography, THE KLF is a trawl through chaos on the trail of a beautiful, accidental mythology."""

# chunk_size: max characters per chunk
# chunk_overlap: overlapping characters to keep context
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len,
)

chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}\n")

Chunk 1: They were the bestselling singles band in the

Chunk 2: in the world. They had awards, credibility,

Chunk 3: commercial success and creative freedom. Then

Chunk 4: Then they deleted their records,

Chunk 5: erased themselves from musical history

Chunk 6: history and burnt their last million pounds in a

Chunk 7: in a boathouse on the Isle of Jura. And they

Chunk 8: And they couldn't say why.

Chunk 9: This is not just the story of The KLF. It

Chunk 10: KLF. It is a book about Carl Jung, Alan Moore,

Chunk 11: Moore, Robert Anton Wilson, Ken Campbell, Dada,

Chunk 12: Dada, Situationism, Discordianism, magic, chaos,

Chunk 13: chaos, punk, rave, the

Chunk 14: alchemical symbolism of Doctor Who and

Chunk 15: Who and the special power of the number 23.

Chunk 16: Wildly unauthorised and unlike any other

Chunk 17: any other music biography, THE KLF is a trawl

Chunk 18: a trawl through chaos on the trail of a

Chunk 19: of a beautiful, accidental mythology.



In [10]:
type(chunks)

list

In [14]:
from pathlib import Path
import json

In [15]:
notebook_dir = Path.cwd()
arxiv_dir = notebook_dir.parent / "raw" / "arxiv"

In [16]:
print(notebook_dir)

c:\Users\rzamb\Documents\rag-eval-stack\notebooks


In [17]:
print(arxiv_dir)

c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv


In [18]:
for file in arxiv_dir.iterdir():  
    if file.is_file():  # Check if it's a file
        print(file)

c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\anti-money_laundering_improvement_act.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\apache_airflow.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\apache_beam.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\apache_kafka.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\apache_spark.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\attention_is_all_you_need.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\bert_language_model.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\capital_one.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\chromadb.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\dask_software.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\data_analysis_for_fraud_detection.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\embedding_machine_learning.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\gpt-4.json
c:\Users\rzamb\Documents\rag-eval-stack\raw\ar

In [42]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    length_function=len,
)

In [ ]:
import os  
from pathlib import Path
import json

notebook_dir = Path.cwd()
arxiv_dir = notebook_dir.parent / "raw" / "arxiv"

arxiv_chunks = []

for root, _, files in tqdm(os.walk(arxiv_dir), desc="Chunking and encoding entries from Arxiv...", colour='green'):
    for filename in files:  # loop through files in the current directory

        file_path = os.path.join(root, filename)
        file_topic = filename.split(".")[0].replace("_", " ")
        
        with open(file_path, 'r') as file:
            
            data = json.load(file)

            for entry in data:

                print(entry.keys())
                # parsed_data = json.dumps(entry, indent=4)
                entry_text = entry['summary']

                print(file_topic)
                print(file_path)
                print(entry_text)

                topic_entry_chunks = text_splitter.split_text(entry_text)

                print("-"*50)

                for i, chunk in enumerate(topic_entry_chunks):
                    arxiv_chunks.append(chunk)
                    print(f"Chunk {i+1}: {chunk}\n")

arxiv_chunk_embeddings = model.encode(
    arxiv_chunks,
    normalize_embeddings=True,  # important for cosine similarity
    show_progress_bar=True
)

Chunking and encoding entries from Arxiv...: 1it [00:00, 115.78it/s]


dict_keys(['title', 'summary', 'url', 'published'])
anti-money laundering improvement act
c:\Users\rzamb\Documents\rag-eval-stack\raw\arxiv\anti-money_laundering_improvement_act.json
In this paper, we focused on using deep learning methods for detecting money laundering in financial transaction networks, in order to demonstrate that it can be used as a complement or instead of the more commonly used rule-based systems and conventional Anti-Money Laundering (AML) systems. The paper explores the pivotal role played by Anti-Money Laundering (AML) activities in the global financial industry. It underscores the drawbacks of conventional AML systems, which exhibit high rates of false positives and lack the sophistication to uncover intricate money laundering schemes. To tackle these challenges, the paper proposes an advanced AML system that capitalizes on link analysis using deep learning techniques. At the heart of this system lies the utilization of centrality algorithms like Degree Centra

Batches: 100%|██████████| 4/4 [00:01<00:00,  3.97it/s]


In [63]:
wiki_dir = notebook_dir.parent / "raw" / "wikipedia"

wiki_chunks = []

for root, _, files in tqdm(os.walk(wiki_dir), desc="Chunking and encoding entries from Wikipedia...", colour='green'):
    for filename in files:  # loop through files in the current directory

        file_path = os.path.join(root, filename)
        file_topic = filename.split(".")[0].replace("_", " ")
        
        with open(file_path, 'r', encoding='utf-8') as file:
            
            entry_text = json.load(file)

            #print(entry)
            #parsed_data = json.dumps(entry, indent=4)
            #entry_text = entry['summary']

            print('topic: ', file_topic)
            print('path: ',file_path)
            print('entry: ', entry_text)

            topic_entry_chunks = text_splitter.split_text(entry_text)

            print("-"*50)

            for i, chunk in enumerate(topic_entry_chunks):
                wiki_chunks.append(chunk)
                print(f"Chunk {i+1}: {chunk}\n")

wiki_chunk_embeddings = model.encode(
    wiki_chunks,
    normalize_embeddings=True,  # important for cosine similarity
    show_progress_bar=True
)

Chunking and encoding entries from Wikipedia...: 1it [00:00, 54.93it/s]


topic:  anti-money laundering improvement act
path:  c:\Users\rzamb\Documents\rag-eval-stack\raw\wikipedia\anti-money_laundering_improvement_act.json
entry:  The Anti-Money Laundering Improvement Act (AML) is a collection of regulations and laws in the United States aimed at combating money laundering and terrorist financing. The act builds upon the Bank Secrecy Act (BSA), the first anti-money laundering enforcement law.

History
Overview of Act
The Anti-Money Laundering Improvement Act established national and international policies to prevent and combat money laundering and terrorist financing.
It protects the integrity of financial institutions by detecting money laundering activities, which involve converting illegally obtained funds into legitimate assets through complex transactions and disguising the proceeds as lawful funds.
The act is part of the Bank Secrecy Act, a specific piece of U.S. legislation focusing on reporting and record-keeping requirements for financial instituti

Batches: 100%|██████████| 25/25 [00:05<00:00,  4.59it/s]


In [ ]:
wiki_chunk_embeddings[0]
wiki_chunks[0]

'The Anti-Money Laundering Improvement Act (AML) is a collection of regulations and laws in the United States aimed at combating money laundering and terrorist financing. The act builds upon the Bank Secrecy Act (BSA), the first anti-money laundering enforcement law.'

In [66]:
len(wiki_chunk_embeddings) == len(wiki_chunks) 

True